In [1]:
# Cell 1
!pip install -q -U transformers peft accelerate bitsandbytes gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 9.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 45.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 5.0 MB/s eta 0:00:00


In [7]:
# Cell 2
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"
# Double-check this path matches your Kaggle Input dataset name
ADAPTER_PATH = "/kaggle/input/datasets/armanhalavi/thesis-instruct-adapter/instruct_adapter" 

print("Configuring 8-bit Quantization for Tensor Cores...")
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True, 
)

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

print("Loading Instruct Base Model across Dual T4s...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" # This splits the model across both GPUs
)

print("Fusing your custom Instruct Adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("✅ 8-Bit Model Loaded.")

Configuring 8-bit Quantization for Tensor Cores...
Loading Tokenizer...
Loading Instruct Base Model across Dual T4s...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Fusing your custom Instruct Adapter...
✅ 8-Bit Model Loaded.


In [11]:
# Cell 3
import gradio as gr

def humanize_draft(draft_text):
    # THE FIX: Banning AI-speak and forcing direct, active-voice translation
    messages = [
        {"role": "system", "content": "You are a direct, minimalist academic researcher in statistics and data science. Your sole job is to rewrite AI-generated drafts into dry, human-sounding academic prose.\n\nCRITICAL RULES:\n1. BAN FLOWERY LANGUAGE: Do not use words like 'burgeoning', 'paradigm', 'proponents', 'detractors', 'posits', 'corroborated', 'conversely', or 'vis-à-vis'.\n2. BE DIRECT: Use simple, active-voice sentence structures. Do not overcomplicate the phrasing.\n3. PRESERVE STRUCTURE: You MUST keep the exact same number of paragraphs as the input. Do not merge paragraphs.\n4. Output ONLY the rewritten text."},
        {"role": "user", "content": f"Rewrite this:\n\n{draft_text}"}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        **inputs,
        max_new_tokens=1500,  # THE FIX: Massive headroom for multi-paragraph generation
        temperature=0.5,      # Keeps the logic grounded in your 52 papers
        top_p=0.9,            
        do_sample=True,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id
    )
    
    input_length = inputs["input_ids"].shape[1]
    response_ids = outputs[0][input_length:]
    final_text = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    
    return final_text

print("\nLaunching Web Interface from Kaggle...")

interface = gr.Interface(
    fn=humanize_draft,
    inputs=gr.Textbox(lines=10, placeholder="Paste your robotic/clunky draft paragraph here...", label="Original Draft"),
    outputs=gr.Textbox(lines=15, label="Humanized Academic Output (8-Bit)"),
    title="Instruct-Driven Thesis Humanizer",
    description="Upgraded token ceiling for multi-paragraph support and forced burstiness to bypass AI tone."
)

interface.launch(share=True, debug=True, theme=gr.themes.Soft())


Launching Web Interface from Kaggle...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c3f09e2cd2a180c5f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 duri

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c3f09e2cd2a180c5f0.gradio.live
